# Honesty-Humility Audit Pipeline

Audit LLM judge quality for the honesty-humility eval (HEXACO-PI-R / SD3 / Dirty Dozen inspired, 6 facets, 36 items).

The honesty-humility eval lives outside niels' `EvalConfig`, so this notebook loads data directly
from the notebook's CSV output rather than going through `bridge.py`.

Steps:
1. Setup & install deps
2. Load honesty-humility results
3. Stratified sampling
4. Run alternative judges
5. Human annotation (optional)
6. Analysis & summary
7. Group-stratified human sample (differential bias test)

## 1. Setup

In [ ]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/junekhunter/spar-ood-propensities /content/repo 2>/dev/null || !git -C /content/repo pull
    %cd /content/repo/june/vibes_audit
    !pip install -q pyyaml pandas numpy scipy scikit-learn tenacity tqdm openai anthropic python-dotenv
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    # Symlink output to Drive for persistence
    _drive = '/content/drive/MyDrive/spar-ood-propensities/june/vibes_audit/output'
    os.makedirs(_drive, exist_ok=True)
    !ln -sfn {_drive} output
    os.environ["OPENROUTER_API_KEY"] = userdata.get("openrouter")
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("anthropic")
else:
    %cd {os.path.dirname(os.path.abspath('__file__')) if '__file__' not in dir() else os.path.dirname(__file__)}

from dotenv import load_dotenv
load_dotenv()
print("Working dir:", os.getcwd())

In [ ]:
from audit_config import from_yaml
from sample_for_review import load_data, stratified_sample
from run_alt_judges import run_judges
from analyze import inter_judge_correlations, bias_probes, audit_summary, gwets_ac2, cohen_weighted_kappa, score_to_bins, confusion_matrix_plot
from pathlib import Path
import pandas as pd
import numpy as np

CONFIG_PATH = Path("configs/honesty_humility.yaml")
DARK_DIR = Path("../dark")
OUTPUT_DIR = Path("output/honesty_humility")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = from_yaml(CONFIG_PATH, output_dir=str(OUTPUT_DIR))
METRIC = config.score_column
print(f"Config: {config.display_name}")
print(f"Metric: {METRIC}")
print(f"Buckets: {[(b.number, b.label) for b in config.buckets]}")

## 2. Load Honesty-Humility Results

Load `results.csv` produced by `honesty_humility_analysis.ipynb`.

In [ ]:
# Load results from honesty_humility_analysis.ipynb
results_path = DARK_DIR / "results.csv"

results_df = pd.read_csv(results_path, low_memory=False)

# Rename answer -> response for audit framework compatibility
if "answer" in results_df.columns and "response" not in results_df.columns:
    results_df = results_df.rename(columns={"answer": "response"})

print(f"Loaded {len(results_df)} rows from {results_path.name}")
print(f"Groups: {results_df['group'].unique().tolist()}")
print(f"Facets: {results_df['facet_name'].unique().tolist()}")
print(f"\nScore stats ({METRIC}):")
print(results_df[METRIC].describe().round(1))

In [ ]:
# Score distributions by group
results_df.groupby('group')[METRIC].describe().round(1)

## 3. Stratified Sampling

In [ ]:
# Save consolidated results and run stratified sampling
data_path = OUTPUT_DIR / "all_results.csv"
results_df.to_csv(data_path, index=False)

config = from_yaml(CONFIG_PATH, data_path=str(data_path), output_dir=str(OUTPUT_DIR))
TARGET_N = 100
config.target_n = min(TARGET_N, len(results_df))

df_loaded = load_data(config)
sample = stratified_sample(df_loaded, config)

print(f"\nSampled: {len(sample)} rows")

# Save full sample
full_path = OUTPUT_DIR / f"sample_{len(sample)}.csv"
sample.to_csv(full_path, index=False)
print(f"Saved: {full_path}")

# Save blind sample (for human annotation)
blind_cols = ["question", "response"] + [
    c for c in config.metadata_columns if c in sample.columns
]
blind = sample[blind_cols].copy()
blind["human_label"] = ""
blind_path = OUTPUT_DIR / f"sample_{len(sample)}_blind.csv"
blind.to_csv(blind_path, index=False)
print(f"Saved: {blind_path}")

In [ ]:
# Preview the sample
sample[["question", "response", METRIC, "group", "facet_name"]].head()

## 4. Run Alternative Judges

Requires `OPENROUTER_API_KEY` and/or `ANTHROPIC_API_KEY` set above.

In [ ]:
sample_df = pd.read_csv(full_path, low_memory=False)
config = from_yaml(CONFIG_PATH, output_dir=str(OUTPUT_DIR))

print(f"Running alt judges on {len(sample_df)} rows...")
print(f"Judges: {[j['name'] for j in config.alt_judges]}")

result = run_judges(sample_df, config)

alt_path = OUTPUT_DIR / "alt_judge_scores.csv"
result.to_csv(alt_path, index=False)
print(f"\nSaved: {alt_path}")

In [ ]:
# Quick correlation check
score_cols = [c for c in result.columns if c.endswith("_score") and c != config.score_column]
for col in score_cols:
    valid = result[col].notna() & result[config.score_column].notna()
    if valid.sum() > 0:
        corr = result.loc[valid, col].corr(result.loc[valid, config.score_column])
        print(f"Correlation {config.score_column} vs {col}: {corr:.3f}")

## 5. Human Annotation (Optional)

The annotation GUI requires a local server. In Colab, you can review samples manually instead.

To use the full GUI locally:
```bash
cd june/vibes_audit
python annotate.py --config configs/honesty_humility.yaml --output-dir output/honesty_humility
```

In [ ]:
# Manual review: inspect a few samples
blind_df = pd.read_csv(blind_path)
for i, row in blind_df.head(5).iterrows():
    print(f"\n{'='*60}")
    print(f"Sample {i} | group={row.get('group', '?')} | facet={row.get('facet_name', '?')}")
    print(f"{'='*60}")
    print(f"Q: {str(row['question'])[:200]}...")
    print(f"\nA: {str(row['response'])[:300]}...")

### 5b. Load Human Annotations

After running the annotation GUI locally, upload or load the annotations CSV.

In [ ]:
# Try to find the most recent annotations file
ann_candidates = sorted(OUTPUT_DIR.glob("*annotations*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
ann_path = ann_candidates[0] if ann_candidates else OUTPUT_DIR / "human_annotations.csv"

if not ann_path.exists():
    print(f"No annotations found at {ann_path}")
    print("Run the annotator locally first:")
    print(f"  python annotate.py --config configs/honesty_humility.yaml --output-dir output/honesty_humility")
else:
    human_df = pd.read_csv(ann_path)
    n_labeled = human_df["human_label"].notna() & (human_df["human_label"] != "")
    print(f"Loaded {n_labeled.sum()} annotations from {ann_path.name}")
    print(f"Label distribution:")
    print(human_df.loc[n_labeled, "human_label"].value_counts())

### Human vs Judge Agreement

Compare human labels against the original judge scores and alt judge scores.

In [ ]:
if ann_path.exists():
    import matplotlib.pyplot as plt
    from scipy import stats

    human_df = pd.read_csv(ann_path)
    config = from_yaml(CONFIG_PATH, output_dir=str(OUTPUT_DIR))

    # Map human labels to numeric (bucket number)
    label_to_num = {b.label: b.number for b in config.buckets}
    label_to_num[config.INCOHERENT_LABEL] = 0

    labeled = human_df[human_df["human_label"].notna() & (human_df["human_label"] != "")].copy()
    labeled["human_score"] = labeled["human_label"].map(label_to_num)
    labeled = labeled.dropna(subset=["human_score"])
    labeled["human_score"] = labeled["human_score"].astype(int)

    # Exclude INCOHERENT for agreement stats
    valid = labeled[labeled["human_score"] > 0].copy()

    # Map judge scores to same bucket scale
    bins = [0, 20, 40, 60, 80, 100]
    bucket_labels = [1, 2, 3, 4, 5]
    valid["judge_bucket"] = pd.cut(valid[METRIC], bins=bins, labels=bucket_labels, include_lowest=True).astype(int)

    # Agreement stats
    exact = (valid["human_score"] == valid["judge_bucket"]).mean()
    within1 = (abs(valid["human_score"] - valid["judge_bucket"]) <= 1).mean()

    rho, rho_p = stats.spearmanr(valid["human_score"], valid[METRIC])
    r, r_p = stats.pearsonr(valid["human_score"], valid[METRIC])

    ac2 = gwets_ac2(valid["human_score"].values, valid["judge_bucket"].values)
    kappa = cohen_weighted_kappa(valid["human_score"].values, valid["judge_bucket"].values)

    n_incoherent = (labeled["human_score"] == 0).sum()

    print(f"Human vs Judge Agreement (n={len(valid)}, {n_incoherent} incoherent excluded)")
    print(f"{'='*60}")
    print(f"  Exact bucket match:  {exact:.1%}")
    print(f"  Within 1 bucket:     {within1:.1%}")
    print(f"  Spearman rho:        {rho:.3f} (p={rho_p:.4f})")
    print(f"  Pearson r:           {r:.3f} (p={r_p:.4f})")
    print(f"  Gwet's AC2:          {ac2:.3f}")
    print(f"  Cohen's weighted K:  {kappa:.3f}")

    # Confusion matrix
    fig = confusion_matrix_plot(
        valid["human_score"].values, valid["judge_bucket"].values,
        labels=bucket_labels,
        title=f"Human vs Judge — {config.display_name}"
    )
    plt.show()

In [ ]:
# Disagreement analysis: which samples did human and judge disagree on most?
if ann_path.exists() and 'valid' in dir() and len(valid) > 0:
    valid["disagreement"] = abs(valid["human_score"] - valid["judge_bucket"])
    disagreed = valid[valid["disagreement"] >= 2].sort_values("disagreement", ascending=False)

    print(f"Large disagreements (>= 2 buckets apart): {len(disagreed)} / {len(valid)}")
    print()

    for i, (_, row) in enumerate(disagreed.head(10).iterrows()):
        print(f"--- Disagreement #{i+1}: human={int(row['human_score'])} vs judge_bucket={int(row['judge_bucket'])} (raw={row[METRIC]:.0f}) ---")
        print(f"Group: {row.get('group', '?')} | Facet: {row.get('facet_name', '?')}")
        print(f"Q: {str(row['question'])[:150]}...")
        print(f"A: {str(row['response'])[:250]}...")
        print()

### Systematic Bias Analysis

Quantify the direction and source of disagreements between human and judge.

In [ ]:
if ann_path.exists() and 'valid' in dir() and len(valid) > 0:
    import matplotlib.pyplot as plt
    from scipy import stats

    # --- Directional bias ---
    valid["delta"] = valid["human_score"] - valid["judge_bucket"]  # positive = judge too low
    mean_delta = valid["delta"].mean()
    judge_too_low = (valid["delta"] > 0).sum()
    judge_too_high = (valid["delta"] < 0).sum()
    exact_agree = (valid["delta"] == 0).sum()

    print("=" * 60)
    print("DIRECTIONAL BIAS (human - judge_bucket)")
    print("=" * 60)
    print(f"  Mean delta:       {mean_delta:+.2f} buckets {'(judge scores too low)' if mean_delta > 0 else '(judge scores too high)'}")
    print(f"  Judge too low:    {judge_too_low} ({judge_too_low/len(valid):.1%})")
    print(f"  Judge too high:   {judge_too_high} ({judge_too_high/len(valid):.1%})")
    print(f"  Exact agreement:  {exact_agree} ({exact_agree/len(valid):.1%})")

    # --- Bias by score region ---
    print(f"\n{'='*60}")
    print("BIAS BY SCORE REGION")
    print("=" * 60)
    for bucket in sorted(valid["judge_bucket"].unique()):
        subset = valid[valid["judge_bucket"] == bucket]
        if len(subset) > 0:
            bucket_label = [b.label for b in config.buckets if b.number == bucket]
            label = bucket_label[0] if bucket_label else str(bucket)
            print(f"  Bucket {bucket} ({label}): mean delta={subset['delta'].mean():+.2f}, n={len(subset)}")

    # --- Plot ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(valid["delta"], bins=range(-5, 6), edgecolor='black', alpha=0.7)
    axes[0].axvline(0, color='red', linestyle='--')
    axes[0].set_xlabel("human - judge (buckets)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Disagreement Distribution")

    axes[1].scatter(valid[METRIC], valid["human_score"], alpha=0.3, s=20)
    axes[1].plot([0, 100], [1, 5], 'r--', alpha=0.5)
    axes[1].set_xlabel(f"Judge {METRIC}")
    axes[1].set_ylabel("Human bucket")
    axes[1].set_title("Human vs Judge Scatter")
    plt.tight_layout()
    plt.show()

In [ ]:
if ann_path.exists() and 'valid' in dir() and len(valid) > 0:
    # --- Breakdown by group ---
    print("=" * 60)
    print("DISAGREEMENT BY GROUP")
    print("=" * 60)

    group_stats = valid.groupby("group").agg(
        n=("delta", "size"),
        mean_delta=("delta", "mean"),
        large_disagree=("disagreement", lambda x: (x >= 2).sum()),
        mean_judge=(METRIC, "mean"),
        mean_human=("human_score", "mean"),
    ).sort_values("mean_delta", ascending=False)
    group_stats["large_disagree_pct"] = (group_stats["large_disagree"] / group_stats["n"] * 100).round(1)
    group_stats = group_stats.round(2)
    print(group_stats.to_string())

    # --- Breakdown by facet ---
    if "facet_name" in valid.columns:
        print(f"\n{'='*60}")
        print("DISAGREEMENT BY FACET")
        print("=" * 60)
        facet_stats = valid.groupby("facet_name").agg(
            n=("delta", "size"),
            mean_delta=("delta", "mean"),
            large_disagree=("disagreement", lambda x: (x >= 2).sum()),
        ).sort_values("mean_delta", ascending=False)
        facet_stats["large_disagree_pct"] = (facet_stats["large_disagree"] / facet_stats["n"] * 100).round(1)
        facet_stats = facet_stats.round(2)
        print(facet_stats.to_string())

## 6. Analysis & Summary

In [ ]:
alt_df = pd.read_csv(alt_path, low_memory=False)
config = from_yaml(CONFIG_PATH, output_dir=str(OUTPUT_DIR))

# Check for human annotations
ann_path_check = OUTPUT_DIR / "human_annotations.csv"
human_df = pd.read_csv(ann_path_check) if ann_path_check.exists() else pd.DataFrame()

# Audit summary
summary = audit_summary(config, human_df, alt_df)
summary["eval"] = "honesty_humility"

summary_path = OUTPUT_DIR / "audit_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"Honesty-Humility Audit Summary:")
print(f"{'='*70}")
for _, row in summary.iterrows():
    icon = {"PASS": "PASS", "MARGINAL": "WARN", "FAIL": "FAIL"}.get(row["Status"], "?")
    print(f"  [{icon}] {row['Metric']}: {row['Value']} (threshold {row['Threshold']})")

In [ ]:
# Bias probes
group_cols = [c for c in config.metadata_columns if c in alt_df.columns]
probes = bias_probes(alt_df, config.score_column, group_cols)

print("Bias Probes:")
print(f"{'='*70}")
for probe_name, res in probes.items():
    if "r" in res:
        print(f"  {probe_name}: r={res['r']:.3f}, p={res['p']:.4f}")
    elif "F" in res:
        print(f"  {probe_name}: F={res['F']:.2f}, p={res['p']:.4f}")
        if "group_means" in res:
            for gname, gmean in res["group_means"].items():
                print(f"    {gname}: mean={gmean:.1f}")

In [ ]:
# Inter-judge correlations
all_score_cols = [config.score_column] + [
    c for c in alt_df.columns if c.endswith("_score") and c != config.score_column
]
corr_df = inter_judge_correlations(alt_df, all_score_cols)
print("Inter-Judge Correlations:")
corr_df

## 7. Group-Stratified Human Sample

The previous audit sample was stratified by score bin. To directly test whether the judge is
differentially biased across groups, we need a sample stratified by **group** with enough power
to detect practically significant bias.

**Design:** Two-sample t-test of (human_score - judge_score) between dark and each control group.
Target MDE=10 as "practically significant."

In [ ]:
from scipy import stats

# --- Estimate sigma from existing audit data ---
ann_path_sigma = sorted(OUTPUT_DIR.glob("*annotations*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)

sigma_est = 20.0  # conservative default

if ann_path_sigma:
    human_df_for_sigma = pd.read_csv(ann_path_sigma[0])
    config_for_sigma = from_yaml(CONFIG_PATH, output_dir=str(OUTPUT_DIR))

    label_to_num = {b.label: b.number for b in config_for_sigma.buckets}
    label_to_num[config_for_sigma.INCOHERENT_LABEL] = 0

    scored = human_df_for_sigma.copy()
    scored["human_score"] = scored["human_label"].map(label_to_num)
    scored = scored.dropna(subset=["human_score"])
    scored = scored[scored["human_score"] > 0]

    if len(scored) > 0:
        # Convert human buckets to 0-100 scale midpoints
        BUCKET_TO_SCORE = {
            "Very Honest-Humble": 90, "Honest-Humble": 70, "Balanced": 50,
            "Dark": 30, "Very Dark": 10,
        }
        scored["human_score_100"] = scored["human_label"].map(BUCKET_TO_SCORE)
        scored["bias"] = scored["human_score_100"] - scored[METRIC]
        sigma_est = scored["bias"].std()
        print(f"Estimated sigma from existing annotations: {sigma_est:.1f}")

# --- Power calculation ---
MDE = 10  # practically significant bias
alpha = 0.05
power_target = 0.80
z_alpha = stats.norm.ppf(1 - alpha / 2)
z_beta = stats.norm.ppf(power_target)
N_PER_GROUP = int(np.ceil(2 * (sigma_est * (z_alpha + z_beta) / MDE) ** 2))

groups = results_df["group"].unique()
TOTAL_N = N_PER_GROUP * len(groups)

print(f"\nPower analysis: MDE={MDE}, sigma={sigma_est:.1f}, alpha={alpha}, power={power_target}")
print(f"  N per group: {N_PER_GROUP}")
print(f"  Groups: {len(groups)}")
print(f"  Total sample: {TOTAL_N}")

In [ ]:
# --- Sample equally from each group ---
results_path = DARK_DIR / "results.csv"
full_df = pd.read_csv(results_path, low_memory=False)

if "answer" in full_df.columns and "response" not in full_df.columns:
    full_df = full_df.rename(columns={"answer": "response"})

groups = full_df["group"].unique().tolist()
group_samples = []
for g in groups:
    pool = full_df[full_df["group"] == g]
    n = min(N_PER_GROUP, len(pool))
    group_samples.append(pool.sample(n=n, random_state=42))
    print(f"  {g}: sampled {n} / {len(pool)}")

group_sample = pd.concat(group_samples, ignore_index=True)

# Save
gs_full_path = OUTPUT_DIR / f"group_sample_{len(group_sample)}.csv"
group_sample.to_csv(gs_full_path, index=False)

gs_blind = group_sample[["question", "response"] + [
    c for c in config.metadata_columns if c in group_sample.columns
]].copy()
gs_blind["human_label"] = ""
gs_blind_path = OUTPUT_DIR / f"group_sample_{len(group_sample)}_blind.csv"
gs_blind.to_csv(gs_blind_path, index=False)

print(f"\nSaved: {gs_full_path}")
print(f"Saved: {gs_blind_path}")
print(f"\nAnnotate with:")
print(f"  python annotate.py --config configs/honesty_humility.yaml --output-dir output/honesty_humility")

In [ ]:
# --- Analysis cell (run after annotation) ---
# Load completed annotations and test for differential bias
# Supports both 0-100 numeric scores (human_score column) and
# bucket labels from the GUI (human_label column -> midpoint conversion)

BUCKET_TO_SCORE = {
    "Very Honest-Humble": 90, "Honest-Humble": 70, "Balanced": 50,
    "Dark": 30, "Very Dark": 10,
}

gs_ann_path = OUTPUT_DIR / f"group_sample_{TOTAL_N}_annotations.csv"
gs_full_path = OUTPUT_DIR / f"group_sample_{TOTAL_N}.csv"

if gs_ann_path.exists():
    ann = pd.read_csv(gs_ann_path)

    # Determine score source: human_score column or bucket labels
    if "human_score" in ann.columns and ann["human_score"].notna().any():
        print("Using numeric human_score column")
    elif "human_label" in ann.columns:
        ann["human_score"] = ann["human_label"].map(BUCKET_TO_SCORE)
        print("Converted bucket labels to midpoint scores")

    # Merge with full scores
    full = pd.read_csv(gs_full_path)
    merged = ann[["human_score", "human_label"]].join(
        full.drop(columns=["human_label"], errors="ignore")
    )

    # Exclude incoherent
    incoherent = ann["human_label"] == config.INCOHERENT_LABEL if hasattr(config, 'INCOHERENT_LABEL') else pd.Series(False, index=ann.index)
    n_incoh = incoherent.sum()
    valid_gs = merged[~incoherent & merged["human_score"].notna()].copy()
    valid_gs["bias"] = valid_gs["human_score"] - valid_gs[METRIC]

    print(f"Converted {len(ann)} bucket labels to midpoint scores")
    print(f"Excluded {n_incoh} INCOHERENT responses")
    print(f"Loaded {len(valid_gs)} annotated samples")

    # --- Differential bias by group ---
    print(f"\n{'='*70}")
    print("DIFFERENTIAL BIAS BY GROUP (human_score - judge_score)")
    print("=" * 70)

    for g in sorted(valid_gs["group"].unique()):
        subset = valid_gs[valid_gs["group"] == g]
        print(f"  {g:40s}: mean bias = {subset['bias'].mean():+.1f} \u00b1 {subset['bias'].sem():.1f} (n={len(subset)})")

    # --- Pairwise tests: is judge differentially biased against dark models? ---
    print(f"\n{'='*70}")
    print("PAIRWISE TESTS: is judge differentially biased against dark models?")
    print("=" * 70)

    # Identify base vs dark groups
    base_groups = [g for g in valid_gs["group"].unique() if "base" in g]
    dark_groups = [g for g in valid_gs["group"].unique() if "dark" in g]

    for dark_g in dark_groups:
        # Find matching base group
        family = dark_g.replace("-dark-v2", "").replace("-dark", "")
        matching_base = [b for b in base_groups if family in b]
        if not matching_base:
            continue
        base_g = matching_base[0]

        dark_bias = valid_gs[valid_gs["group"] == dark_g]["bias"]
        base_bias = valid_gs[valid_gs["group"] == base_g]["bias"]

        delta = dark_bias.mean() - base_bias.mean()
        t_stat, p_val = stats.ttest_ind(dark_bias, base_bias)
        ci = stats.t.interval(0.95, df=len(dark_bias)+len(base_bias)-2,
                              loc=delta, scale=np.sqrt(dark_bias.var()/len(dark_bias) + base_bias.var()/len(base_bias)))

        print(f"  {dark_g} vs {base_g}: \u0394bias = {delta:+.1f} pts, t={t_stat:.2f}, p={p_val:.4f}")
        print(f"    \u2192 {'RULED OUT' if abs(ci[1] - ci[0])/2 < MDE else 'INCONCLUSIVE'}: 95% CI [{ci[0]:.1f}, {ci[1]:.1f}] within \u00b1{MDE}")

    # --- Bias-corrected estimates ---
    print(f"\n{'='*70}")
    print("BIAS-CORRECTED HONESTY-HUMILITY ESTIMATES")
    print("=" * 70)
    print(f" {'Group':40s} {'Judge mean':>12s} {'Mean bias':>12s} {'Corrected':>12s}")
    for g in sorted(valid_gs["group"].unique()):
        subset = valid_gs[valid_gs["group"] == g]
        jmean = subset[METRIC].mean()
        bmean = subset["bias"].mean()
        print(f" {g:40s} {jmean:12.1f} {bmean:12.1f} {jmean + bmean:12.1f}")
else:
    print(f"No group-sample annotations found at {gs_ann_path}")
    print(f"Expected total: {TOTAL_N} rows")
    print("Run annotation first, then re-run this cell.")